In [6]:
import os
import time
import numpy as np
from pathlib import Path
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm

# Разрешаем загрузку усеченных изображений
ImageFile.LOAD_TRUNCATED_IMAGES = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
class ACDCDataset(Dataset):
    def __init__(self, acdc_root, additional_root, split="train", transform=None, 
                 additional_split_ratio=(0.7, 0.15, 0.15)):
        """
        ACDC датасет с дополнительными классами
        """
        self.transform = transform
        self.split = split
        
        # Все классы
        self.classes = ["fog", "night", "rain", "snow", "night_fog", "night_rain", "night_snow"]
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        self.samples = []
        self.corrupted_files = []
        
        # Загрузка старых классов из ACDC
        acdc_path = Path(acdc_root)
        for cls in ["fog", "night", "rain", "snow"]:
            split_path = acdc_path / cls / split
            if split_path.exists():
                for img_path in split_path.rglob("*.png"):
                    # Проверяем, что файл не поврежден
                    try:
                        with Image.open(img_path) as test_img:
                            test_img.load()
                        self.samples.append((img_path, self.class_to_idx[cls]))
                    except Exception as e:
                        self.corrupted_files.append(img_path)
                        print(f"  ⚠️ Поврежден: {img_path}")
        
        # Загрузка и разбиение новых классов
        additional_path = Path(additional_root)
        train_ratio, val_ratio, test_ratio = additional_split_ratio
        
        new_class_images = {cls: [] for cls in ["night_fog", "night_rain", "night_snow"]}
        
        for cls in new_class_images.keys():
            class_path = additional_path / cls
            if class_path.exists():
                for img_path in class_path.rglob("*.png"):
                    # Проверяем, что файл не поврежден
                    try:
                        with Image.open(img_path) as test_img:
                            test_img.load()
                        new_class_images[cls].append(img_path)
                    except Exception as e:
                        self.corrupted_files.append(img_path)
                        print(f"  ⚠️ Поврежден: {img_path}")
        
        # Разбиваем каждый новый класс
        for cls, images in new_class_images.items():
            n_images = len(images)
            print(f"\n{cls}: {n_images} images total")
            
            if n_images == 0:
                continue
            
            if split == "train":
                train_imgs, temp_imgs = train_test_split(
                    images, train_size=train_ratio, random_state=42
                )
                selected_imgs = train_imgs
                
            elif split == "val":
                train_imgs, temp_imgs = train_test_split(
                    images, train_size=train_ratio, random_state=42
                )
                val_imgs, test_imgs = train_test_split(
                    temp_imgs, test_size=test_ratio/(val_ratio+test_ratio), random_state=42
                )
                selected_imgs = val_imgs
                
            else:  # test
                train_imgs, temp_imgs = train_test_split(
                    images, train_size=train_ratio, random_state=42
                )
                val_imgs, test_imgs = train_test_split(
                    temp_imgs, test_size=test_ratio/(val_ratio+test_ratio), random_state=42
                )
                selected_imgs = test_imgs
            
            for img_path in selected_imgs:
                self.samples.append((img_path, self.class_to_idx[cls]))
            
            print(f"  {split}: {len(selected_imgs)} images")
        
        print(f"\n{split} total size: {len(self.samples)}")
        if self.corrupted_files:
            print(f"⚠️ Поврежденных файлов пропущено: {len(self.corrupted_files)}")
        
        # Статистика по классам
        print(f"\nClass distribution in {split}:")
        for cls in self.classes:
            count = sum(1 for _, label in self.samples if label == self.class_to_idx[cls])
            if count > 0:
                print(f"  {cls}: {count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            
            return image, label
        except Exception as e:
            # Если изображение все еще вызывает ошибку, возвращаем заглушку
            print(f"  ❌ Ошибка загрузки {img_path}: {e}")
            dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, label

In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Пути в Kaggle
acdc_root = "/kaggle/input/datasets/ffftory/acdc-weather"
additional_root = "/kaggle/input/datasets/ffftory/acdc-additional/data_weather_final"

# Создаем датасеты
print("Loading datasets...")
train_dataset = ACDCDataset(acdc_root, additional_root, split="train", transform=transform_train)
val_dataset = ACDCDataset(acdc_root, additional_root, split="val", transform=transform_val_test)
test_dataset = ACDCDataset(acdc_root, additional_root, split="test", transform=transform_val_test)

Loading datasets...

night_fog: 123 images total
  train: 86 images

night_rain: 123 images total
  train: 86 images

night_snow: 186 images total
  train: 130 images

train total size: 1902

Class distribution in train:
  fog: 400
  night: 400
  rain: 400
  snow: 400
  night_fog: 86
  night_rain: 86
  night_snow: 130

night_fog: 123 images total
  val: 18 images

night_rain: 123 images total
  val: 18 images

night_snow: 186 images total
  val: 28 images

val total size: 470

Class distribution in val:
  fog: 100
  night: 106
  rain: 100
  snow: 100
  night_fog: 18
  night_rain: 18
  night_snow: 28

night_fog: 123 images total
  test: 19 images

night_rain: 123 images total
  test: 19 images

night_snow: 186 images total
  test: 28 images

test total size: 2066

Class distribution in test:
  fog: 500
  night: 500
  rain: 500
  snow: 500
  night_fog: 19
  night_rain: 19
  night_snow: 28


In [9]:
from torch.utils.data import WeightedRandomSampler

# Получаем все метки из train датасета
train_labels = [label for _, label in train_dataset.samples]

# Считаем количество примеров в каждом классе
class_counts = np.bincount(train_labels)
print("\nOriginal class distribution in train:")
for i, name in enumerate(class_names):
    print(f"  {name}: {class_counts[i]}")

# Вычисляем веса для каждого примера (обратно пропорционально частоте класса)
sample_weights = [1.0 / class_counts[label] for label in train_labels]

# Создаем sampler с возвращением
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # можно увеличить: len(sample_weights) * 2
    replacement=True
)

print(f"\nBalanced sampler created: {len(sample_weights)} samples, {len(sample_weights)} per epoch")



Original class distribution in train:
  fog: 400
  night: 400
  rain: 400
  snow: 400
  night_fog: 86
  night_rain: 86
  night_snow: 130

Balanced sampler created: 1902 samples, 1902 per epoch


In [10]:
# Вычисляем веса классов для loss function
from sklearn.utils.class_weight import compute_class_weight

# Обратите внимание: train_labels уже есть выше, не нужно создавать повторно
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("\nClass weights for loss function (higher weight = underrepresented class):")
class_names = ["fog", "night", "rain", "snow", "night_fog", "night_rain", "night_snow"]
for i, weight in enumerate(class_weights):
    print(f"  {class_names[i]}: {weight:.4f}")


Class weights for loss function (higher weight = underrepresented class):
  fog: 0.6793
  night: 0.6793
  rain: 0.6793
  snow: 0.6793
  night_fog: 3.1595
  night_rain: 3.1595
  night_snow: 2.0901


In [13]:
# ========== DATALOADERS ==========
batch_size = 16

# ВАЖНО: используем sampler вместо shuffle!
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    sampler=sampler,  # ← вот здесь sampler, а не shuffle=True
    num_workers=2, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=2, 
    pin_memory=True
)

In [14]:
# Загружаем предобученную модель ResNet18
model = models.resnet18(weights="IMAGENET1K_V1")

# Изменяем выходной слой на 7 классов с Dropout для регуляризации
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),           # Dropout для предотвращения переобучения
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 7)
)

model = model.to(device)

# Используем weighted loss
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print(f"\nModel classes: 7")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")


Model classes: 7
Total parameters: 11341639


In [15]:
# Обучение
num_epochs = 5
best_val_f1 = 0

for epoch in range(num_epochs):
    print(f"\n===== EPOCH {epoch+1}/{num_epochs} =====")
    
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    start_time = time.time()
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    all_val_preds = []
    all_val_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Метрики
    val_prec = precision_score(all_val_labels, all_val_preds, average="macro", zero_division=0)
    val_rec = recall_score(all_val_labels, all_val_preds, average="macro", zero_division=0)
    val_f1 = f1_score(all_val_labels, all_val_preds, average="macro", zero_division=0)
    
    epoch_time = time.time() - start_time
    
    print(f"\nTrain Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"Val Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1: {val_f1:.4f}")
    print(f"Epoch time: {epoch_time:.2f}s")
    
    # Learning rate scheduling
    scheduler.step(avg_val_loss)
    
    # Сохраняем лучшую модель по F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_weather_model.pth')
        print(f"✓ New best model saved! (val_f1: {val_f1:.4f})")


===== EPOCH 1/5 =====


Validation: 100%|██████████| 30/30 [00:14<00:00,  2.08it/s]



Train Loss: 1.1219, Train Acc: 41.80%
Val Loss: 1.3849, Val Acc: 60.00%
Val Precision: 0.6311, Recall: 0.7114, F1: 0.6190
Epoch time: 80.02s
✓ New best model saved! (val_f1: 0.6190)

===== EPOCH 2/5 =====


Validation: 100%|██████████| 30/30 [00:14<00:00,  2.10it/s]



Train Loss: 0.4848, Train Acc: 75.55%
Val Loss: 0.3485, Val Acc: 95.32%
Val Precision: 0.9607, Recall: 0.9584, F1: 0.9584
Epoch time: 78.79s
✓ New best model saved! (val_f1: 0.9584)

===== EPOCH 3/5 =====


...


Train Loss: 0.2158, Train Acc: 92.48%
Val Loss: 0.1821, Val Acc: 96.38%
Val Precision: 0.9578, Recall: 0.9619, F1: 0.9587
Epoch time: 78.97s
✓ New best model saved! (val_f1: 0.9587)

===== EPOCH 4/5 =====


Validation: 100%|██████████| 30/30 [00:14<00:00,  2.09it/s]



Train Loss: 0.1211, Train Acc: 96.42%
Val Loss: 0.0818, Val Acc: 98.72%
Val Precision: 0.9752, Recall: 0.9813, F1: 0.9781
Epoch time: 79.70s
✓ New best model saved! (val_f1: 0.9781)

===== EPOCH 5/5 =====


Validation: 100%|██████████| 30/30 [00:14<00:00,  2.08it/s]


Train Loss: 0.1171, Train Acc: 96.32%
Val Loss: 0.0835, Val Acc: 98.72%
Val Precision: 0.9676, Recall: 0.9740, F1: 0.9698
Epoch time: 79.50s


In [16]:
# Тестирование лучшей модели
print("\n" + "="*50)
print("TESTING BEST MODEL")
print("="*50)

model.load_state_dict(torch.load('best_weather_model.pth'))
model.eval()

all_test_preds = []
all_test_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu()
        
        all_test_preds.extend(preds.numpy())
        all_test_labels.extend(labels.numpy())

# Основные метрики
test_acc = accuracy_score(all_test_labels, all_test_preds)
test_prec = precision_score(all_test_labels, all_test_preds, average="macro", zero_division=0)
test_rec = recall_score(all_test_labels, all_test_preds, average="macro", zero_division=0)
test_f1 = f1_score(all_test_labels, all_test_preds, average="macro", zero_division=0)

print("\nTEST RESULTS:")
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision (macro): {test_prec:.4f}")
print(f"Recall (macro): {test_rec:.4f}")
print(f"F1-score (macro): {test_f1:.4f}")

# Детальный отчет по классам
class_names = ["fog", "night", "rain", "snow", "night_fog", "night_rain", "night_snow"]
print("\nClassification Report:")
print(classification_report(all_test_labels, all_test_preds, 
                           target_names=class_names, zero_division=0))


TESTING BEST MODEL


Testing: 100%|██████████| 130/130 [01:00<00:00,  2.16it/s]


TEST RESULTS:
Accuracy: 0.9608
Precision (macro): 0.8654
Recall (macro): 0.9359
F1-score (macro): 0.8911

Classification Report:
              precision    recall  f1-score   support

         fog       0.98      0.96      0.97       500
       night       1.00      0.99      1.00       500
        rain       0.95      0.95      0.95       500
        snow       0.97      0.95      0.96       500
   night_fog       0.37      0.74      0.49        19
  night_rain       0.86      1.00      0.93        19
  night_snow       0.93      0.96      0.95        28

    accuracy                           0.96      2066
   macro avg       0.87      0.94      0.89      2066
weighted avg       0.97      0.96      0.96      2066

